# Physics-informed neural networks: A deep learning framework for solving forward and inverse problems involving nonlinear partial differential equations

**Paper:** Raissi, M., Perdikaris, P., Karniadakis, G.E. (2019). *Physics-informed neural networks: A deep learning framework for solving forward and inverse problems involving nonlinear partial differential equations.* Journal of Computational Physics, 378, 686-707.

**Carpeta origen:** `PINNs/1. mecanica de fluidos/Physics-informed neural networks A deep learning framework for solving forward and inverse problems involving nonlinear partial differential equations.pdf`

*(Nota: el archivo de este cuaderno se guarda con un nombre acortado para evitar el limite de longitud de ruta de Windows (260 caracteres); el PDF de origen conserva el titulo completo.)*

## Como se usan las PINNs en este paper

Este es el **paper fundacional** que acuna el termino *Physics-Informed Neural Networks (PINNs)*. Formula el problema general (Eq. 1-2):

$$u_t + \mathcal{N}[u;\lambda]=0,\qquad x\in\Omega,\ t\in[0,T]$$

donde $\mathcal{N}[\cdot;\lambda]$ es un operador diferencial no lineal parametrizado por $\lambda$. Como ejemplo motivador explicito del propio paper (discusion de la Eq. 1): la **ecuacion de Burgers 1D** corresponde a $\mathcal{N}[u;\lambda]=\lambda_1 u u_x - \lambda_2 u_{xx}$.

El metodo (**modelo de tiempo continuo**, Seccion 3.1) aproxima $u(t,x)$ con una red neuronal, define $f(t,x):=u_t+\mathcal{N}[u]$ (Eq. 3) &mdash; que es la misma red evaluada con diferenciacion automatica a traves del operador $\mathcal{N}$ &mdash; y entrena ambas con la perdida compartida (Eq. 4):

$$MSE = MSE_u + MSE_f,\quad MSE_u=\frac{1}{N_u}\sum_i|u(t_u^i,x_u^i)-u^i|^2,\quad MSE_f=\frac{1}{N_f}\sum_i|f(t_f^i,x_f^i)|^2$$

donde $MSE_u$ ajusta los datos iniciales/de contorno y $MSE_f$ impone la EDP en los puntos de colocacion. Este cuaderno reproduce fielmente este framework aplicado al **benchmark de Burgers viscosa** ($\nu=0.01/\pi$), el ejemplo mas emblematico asociado a este paper (usado extensamente en el repositorio oficial):

$$u_t + u\,u_x - \frac{0.01}{\pi}u_{xx}=0,\quad x\in[-1,1],\ t\in[0,1]$$
$$u(0,x)=-\sin(\pi x),\qquad u(t,-1)=u(t,1)=0$$

con la arquitectura estandar del paper: red totalmente conectada, activacion tanh, entrenada con Adam seguido de L-BFGS (Seccion 3.1, parrafo final).

## Repositorio publico

El paper **incluye explicitamente** el enlace a su repositorio oficial en el propio texto (Seccion 2): "All code and data-sets accompanying this manuscript are available on GitHub at https://github.com/maziarraissi/PINNs". Este es el repositorio de referencia canonico de PINNs, y contiene la implementacion original (TensorFlow 1.x) de este mismo ejemplo de Burgers en `appendix/continuous_time_inference (Burgers)/`.

In [ ]:
# Instalacion de dependencias (ejecutar si no estan ya instaladas en el entorno)
%pip install -q torch numpy matplotlib

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(1234)
np.random.seed(1234)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Datos iniciales/de contorno y puntos de colocacion (Eq. 4)

In [ ]:
nu = 0.01 / np.pi

N_u = 100   # puntos de datos iniciales + de contorno (N_u en el paper)
N_f = 10000  # puntos de colocacion para el residuo fisico (N_f en el paper)

# Condicion inicial: u(0,x) = -sin(pi x), x in [-1,1]
x_ic = np.random.uniform(-1, 1, (N_u // 2, 1))
t_ic = np.zeros_like(x_ic)
u_ic = -np.sin(np.pi * x_ic)

# Condiciones de contorno: u(t,-1)=u(t,1)=0
t_bc = np.random.uniform(0, 1, (N_u // 4, 1))
x_bc_left = -np.ones_like(t_bc)
x_bc_right = np.ones_like(t_bc)

tx_u = np.vstack([
    np.hstack([t_ic, x_ic]),
    np.hstack([t_bc, x_bc_left]),
    np.hstack([t_bc, x_bc_right]),
])
u_train = np.vstack([u_ic, np.zeros_like(t_bc), np.zeros_like(t_bc)])

# Puntos de colocacion para el residuo f(t,x), Latin-Hypercube-like (muestreo uniforme aqui)
t_f = np.random.uniform(0, 1, (N_f, 1))
x_f = np.random.uniform(-1, 1, (N_f, 1))
tx_f = np.hstack([t_f, x_f])

tx_u_t = torch.tensor(tx_u, dtype=torch.float32, device=device)
u_train_t = torch.tensor(u_train, dtype=torch.float32, device=device)
tx_f_t = torch.tensor(tx_f, dtype=torch.float32, device=device, requires_grad=True)

## 2. Red PINN (Seccion 2: redes feedforward simples, activacion tanh)

In [ ]:
class PINN(nn.Module):
    def __init__(self, n_hidden=8, n_neurons=20):
        super().__init__()
        layers = [nn.Linear(2, n_neurons), nn.Tanh()]
        for _ in range(n_hidden - 1):
            layers += [nn.Linear(n_neurons, n_neurons), nn.Tanh()]
        layers += [nn.Linear(n_neurons, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, tx):
        return self.net(tx)


model = PINN().to(device)


def f_residual(model, tx):
    """f(t,x) := u_t + u*u_x - nu*u_xx, Eq. (3) especializada a Burgers."""
    u = model(tx)
    grads = torch.autograd.grad(u, tx, grad_outputs=torch.ones_like(u),
                                 create_graph=True, retain_graph=True)[0]
    u_t, u_x = grads[:, 0:1], grads[:, 1:2]
    u_xx = torch.autograd.grad(u_x, tx, grad_outputs=torch.ones_like(u_x),
                                create_graph=True, retain_graph=True)[0][:, 1:2]
    return u_t + u * u_x - nu * u_xx

## 3. Perdida (Eq. 4): $MSE=MSE_u+MSE_f$

In [ ]:
def compute_loss(model):
    u_pred = model(tx_u_t)
    mse_u = torch.mean((u_pred - u_train_t)**2)
    f_pred = f_residual(model, tx_f_t)
    mse_f = torch.mean(f_pred**2)
    return mse_u + mse_f, mse_u.item(), mse_f.item()

## 4. Entrenamiento: Adam seguido de L-BFGS (Seccion 3.1, tal como en el repositorio oficial)

In [ ]:
opt_adam = torch.optim.Adam(model.parameters(), lr=1e-3)
history = []
for epoch in range(5000):
    opt_adam.zero_grad()
    loss, mse_u, mse_f = compute_loss(model)
    loss.backward()
    opt_adam.step()
    history.append(loss.item())
    if epoch % 1000 == 0:
        print(f'[Adam] epoch {epoch:5d} | loss={loss.item():.4e} | MSE_u={mse_u:.4e} | MSE_f={mse_f:.4e}')

opt_lbfgs = torch.optim.LBFGS(model.parameters(), lr=1.0, max_iter=500,
                               history_size=50, line_search_fn='strong_wolfe')

def closure():
    opt_lbfgs.zero_grad()
    loss, _, _ = compute_loss(model)
    loss.backward()
    return loss

loss = opt_lbfgs.step(closure)
history.append(loss.item())
print(f'[L-BFGS] final loss={loss.item():.4e}')

## 5. Resultados: superficie $u(t,x)$ y perfiles en instantes fijos (cf. estilo Fig. 1 del paper)

In [ ]:
n_side = 100
ts = np.linspace(0, 1, n_side)
xs = np.linspace(-1, 1, n_side)
Tt, Xx = np.meshgrid(ts, xs)
tx_grid = torch.tensor(np.stack([Tt.ravel(), Xx.ravel()], axis=1), dtype=torch.float32, device=device)
with torch.no_grad():
    u_grid = model(tx_grid).cpu().numpy().reshape(n_side, n_side)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
im = axes[0].pcolormesh(Tt, Xx, u_grid, cmap='rainbow', shading='auto')
axes[0].set_xlabel('t'); axes[0].set_ylabel('x')
axes[0].set_title('u(t,x) predicho por la PINN (Burgers, nu=0.01/pi)')
plt.colorbar(im, ax=axes[0])

for t_snap in [0.25, 0.5, 0.75]:
    tx_line = torch.tensor(np.stack([np.full_like(xs, t_snap), xs], axis=1),
                            dtype=torch.float32, device=device)
    with torch.no_grad():
        u_line = model(tx_line).cpu().numpy().flatten()
    axes[1].plot(xs, u_line, label=f't={t_snap}')
axes[1].set_xlabel('x'); axes[1].set_ylabel('u(t,x)')
axes[1].set_title('Perfiles de u en distintos instantes')
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 4))
plt.semilogy(history)
plt.xlabel('Iteracion'); plt.ylabel('Loss (escala log)')
plt.title('Convergencia (Adam -> L-BFGS)')
plt.grid(alpha=0.3)
plt.show()

Se espera observar el desarrollo del **choque pronunciado** en $x=0$ a medida que avanza el tiempo, caracteristico de la ecuacion de Burgers con viscosidad pequena ($\nu=0.01/\pi$) &mdash; el paper reporta un error relativo $\mathbb{L}_2$ de $6.7\times10^{-4}$ para este benchmark exacto. Para los ejemplos adicionales del paper (Schrodinger no lineal, Allen-Cahn en tiempo discreto con esquemas Runge-Kutta implicitos, y el problema inverso de descubrimiento de $\lambda_1,\lambda_2$ en Navier-Stokes), ver el [repositorio oficial](https://github.com/maziarraissi/PINNs).